Gate 3

# Model experiments

---

In [6]:
#Imports
#from google.colab import drive
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.base import clone
from sklearn.model_selection import KFold, cross_val_score
import platform 
import sklearn
import xgboost
import time

In [7]:
# =============================================================================
# COLAB: Uncomment these lines to load from your Google Drive
# =============================================================================

# drive.mount("/content/drive")
# df = pd.read_csv("/content/drive/MyDrive/allstate_claims_data.csv")

# =============================================================================
# LOCALLY: Load CSV from any repo
# =============================================================================
# Load the dataset from this repo, on any computer.
# Walk up from the current folder until data/allstate_claims_data.csv is found.

data_path = None
for folder in [Path.cwd(), *Path.cwd().resolve().parents]:
    candidate = folder / "data" / "allstate_claims_data.csv"
    if candidate.is_file():
        data_path = candidate
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find data/allstate_claims_data.csv. "
        "Open this notebook from inside the cloned repository."
    )

df = pd.read_csv(data_path)

# Gate 3 - Part 1

Compare at least two approved models against both baselines using the frozen split and original-unit MAE. Tune only on the training data, document all experiments, complete one evidence-driven iteration, and summarize performance, error patterns, reliability, and trade-offs.

Name the model from the validation scores. The named model is the lowest validation MAE on the 28,248 validation claims. The large-claim check is reported beside that name.



**Manifest:** `artifacts/october/gate1-issue-reproducible-project/splits/gate1_partition_manifest.csv`

**Seed:** `random_state=1`

**Inputs:** 130 fields in `artifacts/october/gate1-issue-reproducible-project/field_roles.csv`. `id` and `loss` stay out.

**Models:** random forest, and XGBoost with `reg:absoluteerror`

**Open sign-off:** September Gate 4 and October Gate 1 still have no teammate sign-off. This comparison uses the committed plan.

In [8]:
# Purpose: attach the frozen Gate 1 split to the claims and prove the
# test claims never enter the table used for fitting and validation.
# This cell does not train a model and does not read test loss.

# The manifest is one row per claim id, with partition equal to
# train, validation, or test. It was written in Gate 1 with random_state=1.
# parents[1] steps from data/allstate_claims_data.csv up to the repo root.
manifest_path = (
    data_path.parents[1]
    / "artifacts/october/gate1-issue-reproducible-project/splits/gate1_partition_manifest.csv"
)
manifest = pd.read_csv(manifest_path)

# Drop the test rows before the merge. Those ids stay in `manifest`
# only, so they can be counted without joining their features or loss.
fit_manifest = manifest.loc[manifest["partition"].isin(["train", "validation"])]

# Inner join keeps a claim only when its id is in both the file and the
# train/validation manifest. validate="one_to_one" stops the cell if an
# id is duplicated on either side, which would double-count a claim.
joint = df.merge(fit_manifest, on="id", how="inner", validate="one_to_one")

# These four numbers are the frozen split. Training is what the models
# fit on. Validation is what names a score. Test is held out for November.
train_count = int((joint["partition"] == "train").sum())
validation_count = int((joint["partition"] == "validation").sum())
test_ids = set(manifest.loc[manifest["partition"] == "test", "id"])

# A count above 0 would mean a test claim slipped into the modeling table.
test_ids_in_joint = len(test_ids.intersection(set(joint["id"])))

# Each print is a check against the committed split, not a new calculation
# of how the data should be divided.
print(f"The expected value for training claims is 131822. The actual value for training claims is {train_count}.")
print(f"The expected value for validation claims is 28248. The actual value for validation claims is {validation_count}.")
print(f"The expected value for test claims is 28248. The actual value for test claims is {len(test_ids)}.")
print(f"The expected value for test ids inside the modeling table is 0. The actual value for test ids inside the modeling table is {test_ids_in_joint}.")

The expected value for training claims is 131822. The actual value for training claims is 131822.
The expected value for validation claims is 28248. The actual value for validation claims is 28248.
The expected value for test claims is 28248. The actual value for test claims is 28248.
The expected value for test ids inside the modeling table is 0. The actual value for test ids inside the modeling table is 0.


# Constant baselines

Recompute the training-mean and training-median predictions from training `loss` only, then score both with validation MAE.

In [9]:
# Constants come from training loss only. Each one is predicted for every validation claim.
train_df = joint.loc[joint["partition"] == "train"].copy()
val_df = joint.loc[joint["partition"] == "validation"].copy()

training_mean = float(train_df["loss"].mean())
training_median = float(train_df["loss"].median())
mean_predictions = np.full(len(val_df), training_mean)
median_predictions = np.full(len(val_df), training_median)

mean_mae = mean_absolute_error(val_df["loss"], mean_predictions)
median_mae = mean_absolute_error(val_df["loss"], median_predictions)

print(f"The expected value for the training-mean validation MAE is 1978.3970470120255. The actual value for the training-mean validation MAE is {mean_mae}.")
print(f"The expected value for the training-median validation MAE is 1827.3136338855847. The actual value for the training-median validation MAE is {median_mae}.")
print(f"The expected value for validation claims is 28248. The actual value for validation claims is {len(val_df)}.")

The expected value for the training-mean validation MAE is 1978.3970470120255. The actual value for the training-mean validation MAE is 1978.3970470120253.
The expected value for the training-median validation MAE is 1827.3136338855847. The actual value for the training-median validation MAE is 1827.3136338855847.
The expected value for validation claims is 28248. The actual value for validation claims is 28248.


# Training-only preprocessing

Builds the model inputs from the 130 fields in `field_roles.csv`. On the training rows only, fill each continuous column with its training median, fill each category with its training mode, and one-hot encode the categories so a level absent from training becomes zeros. Apply that fitted pipeline to the validation rows. Leave the continuous fields unscaled, keep every input column, and leave `loss` out of the encoding.

In [10]:
# Use the 130 model inputs. Leave id and loss out of the matrix.
roles_path = (
    data_path.parents[1]
    / "artifacts/october/gate1-issue-reproducible-project/field_roles.csv"
)
roles = pd.read_csv(roles_path)

model_input = roles["model_input"].astype(str).str.lower().eq("true")

feature_roles = roles.loc[model_input].copy() # creates a dataframe of the features that are model inputs
numeric_features = feature_roles.loc[feature_roles["role"] == "continuous", "column"].tolist() # creates a list of the continuous features
categorical_features = feature_roles.loc[feature_roles["role"] == "categorical", "column"].tolist() # creates a list of the categorical features
features = feature_roles["column"].tolist() # creates a list of all the features

print(f"The expected value for model inputs is 130. The actual value for model inputs is {len(features)}.")
print(f"The expected value for continuous inputs is 14. The actual value for continuous inputs is {len(numeric_features)}.")
print(f"The expected value for categorical inputs is 116. The actual value for categorical inputs is {len(categorical_features)}.")

X_train_raw = train_df[features] 
X_val_raw = val_df[features]
y_train = train_df["loss"].to_numpy()
y_val = val_df["loss"].to_numpy()

# fills a blank in any of the cont1 - cont14 featureswith the median of that column on the training claims
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
]) 

# handles cat1 through cat116
# fills a blank in any of the cat1 - cat116 features with the mode of that column on the training claims
# one-hot encodes the categorical features so a level absent from training becomes zeros
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# applies the numeric and categorical pipelines to the features
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train = preprocessor.fit_transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)

print(f"The expected value for processed training rows is 131822. The actual value for processed training rows is {X_train.shape[0]}.")
print(f"The expected value for processed validation rows is 28248. The actual value for processed validation rows is {X_val.shape[0]}.")
print(f"The expected value for matching processed column counts is {X_train.shape[1]}. The actual value for processed validation columns is {X_val.shape[1]}.")

The expected value for model inputs is 130. The actual value for model inputs is 130.
The expected value for continuous inputs is 14. The actual value for continuous inputs is 14.
The expected value for categorical inputs is 116. The actual value for categorical inputs is 116.
The expected value for processed training rows is 131822. The actual value for processed training rows is 131822.
The expected value for processed validation rows is 28248. The actual value for processed validation rows is 28248.
The expected value for matching processed column counts is 1119. The actual value for processed validation columns is 1119.


# Training-only tuning rule

Tune inside the 131,822 training claims with 5 folds and `random_state=1`. Validation and test claims stay outside those folds. Choose the settings for the first fit from that training procedure, then look at validation MAE after the fit.

#  Model Settings

Writes the random forest and XGBoost settings before either model is fit. The forest uses 100 trees, depth 12, and at least 20 training claims per leaf. XGBoost uses absolute error, 200 rounds, depth 6, and a learning rate of 0.1. Both keep `random_state=1`.

In [11]:
# Creates a dictionary of the random forest settings
#These settings are used to create the random forest model
forest_settings = {
    "n_estimators": 100,  # Number of trees in the forest
    "max_depth": 12,      # Maximum depth of the trees
    "min_samples_leaf": 20,  # Minimum number of training claims per leaf
    "random_state": 1      # Random seed for reproducibility
}

# Creates a dictionary of the XGBoost settings
#These settings are used to create the XGBoost model
xgboost_settings = {
    "objective": "reg:absoluteerror", # Objective function for XGBoost
    "n_estimators": 200, # Number of boosting rounds
    "max_depth": 6, # Maximum depth of the trees
    "learning_rate": 0.1, # Learning rate for the model
    "random_state": 1 # Random seed for reproducibility
}

print(f"Forest settings written before the model is fit: {forest_settings}.")
print(f"XGBoost settings written before the model is fit: {xgboost_settings}.")



Forest settings written before the model is fit: {'n_estimators': 100, 'max_depth': 12, 'min_samples_leaf': 20, 'random_state': 1}.
XGBoost settings written before the model is fit: {'objective': 'reg:absoluteerror', 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1, 'random_state': 1}.


# Fit and score the two models

Score each model with 5 folds inside the training claims, then fit it on all 131,822 training claims and record the validation MAE.

In [12]:

# 5 KFolds cut the training claims into 5 folds
cv = KFold(n_splits=5, shuffle=True, random_state=1)

#refits the training-only pipeline inside each fold, then score the 5 validation MAEs
forest_pipe = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestRegressor(**forest_settings, n_jobs=-1)),
])

forest_cv_start = time.perf_counter()
forest_fold_mae = -cross_val_score(
    forest_pipe,
    X_train_raw,
    y_train,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)
forest_cv_seconds = time.perf_counter() - forest_cv_start

# The mean of the 5 validation MAEs is the training-only validation MAE
forest_cv_mae = float(forest_fold_mae.mean())

forest_fit_start = time.perf_counter()
forest_pipe.fit(X_train_raw, y_train)
forest_fit_seconds = time.perf_counter() - forest_fit_start
forest_val_mae = float(mean_absolute_error(y_val, forest_pipe.predict(X_val_raw)))

print(f"The 5-fold training MAE for the random forest is {forest_cv_mae}.")
print(f"The validation MAE for the random forest is {forest_val_mae}.")
print(f"The expected value for validation claims is 28248. The actual value for validation claims is {len(val_df)}.")


xgboost_pipe = Pipeline([
    ("preprocessor", clone(preprocessor)),  
    ("model", XGBRegressor(**xgboost_settings, n_jobs=-1)),
])

xgboost_cv_start = time.perf_counter()
xgboost_fold_mae = -cross_val_score(
    xgboost_pipe,
    X_train_raw,
    y_train,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)
xgboost_cv_seconds = time.perf_counter() - xgboost_cv_start

xgboost_cv_mae = float(xgboost_fold_mae.mean())

xgboost_fit_start = time.perf_counter()
xgboost_pipe.fit(X_train_raw, y_train)
xgboost_fit_seconds = time.perf_counter() - xgboost_fit_start
xgboost_val_mae = float(mean_absolute_error(y_val, xgboost_pipe.predict(X_val_raw)))


print(f"The 5-fold training MAE for the XGBoost is {xgboost_cv_mae}.")
print(f"The validation MAE for the XGBoost is {xgboost_val_mae}.")
print(f"The expected value for validation claims is 28248. The actual value for validation claims is {len(val_df)}.")

The 5-fold training MAE for the random forest is 1282.8323805287268.
The validation MAE for the random forest is 1291.5912066221533.
The expected value for validation claims is 28248. The actual value for validation claims is 28248.
The 5-fold training MAE for the XGBoost is 1162.9594067528374.
The validation MAE for the XGBoost is 1171.71898178563.
The expected value for validation claims is 28248. The actual value for validation claims is 28248.


In [13]:
# Record the software verisions used for these fits.

print(f"The Python version is {platform.python_version()}.")
print(f"The pandas version is {pd.__version__}.")
print(f"The NumPy version is {np.__version__}.")
print(f"The scikit-learn version is {sklearn.__version__}.")
print(f"The XGBoost version is {xgboost.__version__}.")

The Python version is 3.12.4.
The pandas version is 3.0.6.
The NumPy version is 2.5.3.
The scikit-learn version is 1.9.1.
The XGBoost version is 3.4.1.


In [14]:
# Create a directory for the experiment register.
register_dir = (
    data_path.parents[1]
    / "artifacts/october/gate3-issue-reproducible-project/experiments"
)
register_dir.mkdir(parents=True, exist_ok=True)
register_path = register_dir / "experiment_register.csv"

versions = (
    f"Python: {platform.python_version()}\n"
    f"pandas: {pd.__version__}\n"
    f"NumPy: {np.__version__}\n"
    f"scikit-learn: {sklearn.__version__}\n"
    f"XGBoost: {xgboost.__version__}\n"
)

# Creates a string for the feature set
feature_set = "All 130 model inputs in field_roles.csv. id and loss stay out."
preprocessing =(
    "Training-only median fill for numeric fields;"
    "Training-only mode fill and one-hot encoding with handle_unknown=ignore;"
)

# Leave runtime blank when the scoring cell has not stored the timers yet.
def recorded_runtime(cv_name, fit_name):
    if cv_name in globals() and fit_name in globals():
        return globals()[cv_name] + globals()[fit_name]
    return ""

# Creates a list of dictionaries for the constant baselines
rows = [
    {
        "run_id": "constant-mean",
        "hypothesis": "Required constant comparator",
        "changed_variable": "none",
        "feature_set": "none",
        "preprocessing": "none",
        "settings": f"constant={training_mean}",
        "seed": 1,
        "versions": versions,
        "runtime_seconds": 0,
        "train_or_cv_mae": "",
        "validation_mae": mean_mae,
        "validation_claims": 28248,
        "result": "scored",
        "interpretation": "Training mean predicted for every validation claim.",
        "next_decision": "Keep as the required baseline.",
    },
    {
        "run_id": "constant-median",
        "hypothesis": "Constant that minimizes absolute error",
        "changed_variable": "none",
        "feature_set": "none",
        "preprocessing": "none",
        "settings": f"constant={training_median}",
        "seed": 1,
        "versions": versions,
        "runtime_seconds": 0,
        "train_or_cv_mae": "",
        "validation_mae": median_mae,
        "validation_claims": 28248,
        "result": "scored",
        "interpretation": "Training median predicted for every validation claim.",
        "next_decision": "Keep as the companion baseline.",
    },
]

#
rows.extend([
    {
        "run_id": "forest-1",
        "hypothesis": "A tree ensemble can represent non-linear relationships among the anonymous fields",
        "changed_variable": "none",
        "feature_set": feature_set,
        "preprocessing": preprocessing,
        "settings": str(forest_settings),
        "seed": 1,
        "versions": versions,
        "runtime_seconds": recorded_runtime("forest_cv_seconds", "forest_fit_seconds"),
        "train_or_cv_mae": forest_cv_mae,
        "validation_mae": forest_val_mae,
        "validation_claims": len(y_val),
        "result": "scored",
        "interpretation": "First forest settings. 5-fold MAE is the training score.",
        "next_decision": "Sprint 3 tests one change: raise forest n_estimators from 100 to 200. The forest 5-fold MAE is 1282.83, about 120 dollars above the XGBoost 5-fold MAE of 1162.96.",
    },
    {
        "run_id": "xgboost-1",
        "hypothesis": "Gradient boosting aimed at absolute error can lower MAE relative to a constant",
        "changed_variable": "none",
        "feature_set": feature_set,
        "preprocessing": preprocessing,
        "settings": str(xgboost_settings),
        "seed": 1,
        "versions": versions,
        "runtime_seconds": recorded_runtime("xgboost_cv_seconds", "xgboost_fit_seconds"),
        "train_or_cv_mae": xgboost_cv_mae,
        "validation_mae": xgboost_val_mae,
        "validation_claims": len(y_val),
        "result": "scored",
        "interpretation": "First XGBoost settings. 5-fold MAE is the training score.",
        "next_decision": "No setting change. Sprint 3 changes only the forest tree count.",
    },
])

# Saves the lists of dictionaries into a table with collumn names by those keys
register = pd.DataFrame(rows)
register.to_csv(register_path, index=False)

print(f"The expected value for register rows is 4 when both models finish. The actual value for register rows is {len(register)}.")
print(f"The register file is {register_path}.")
print(register[["run_id", "train_or_cv_mae", "validation_mae", "result"]].to_string(index=False))


The expected value for register rows is 4 when both models finish. The actual value for register rows is 4.
The register file is /Users/nevaehdickerson/Documents/MyGitHub/Allstate-1D-predicting-auto-claims-severity/artifacts/october/gate3-issue-reproducible-project/experiments/experiment_register.csv.
         run_id train_or_cv_mae  validation_mae result
  constant-mean                     1978.397047 scored
constant-median                     1827.313634 scored
       forest-1     1282.832381     1291.591207 scored
      xgboost-1     1162.959407     1171.718982 scored


# One complexity check

Compare each model's in-sample training MAE with its 5-fold MAE. The training score is far below the fold score when it is at least 10 percent lower. It is higher when it is worse than the fold score. Otherwise the two scores are close.

When a score is far below, refit that one model with a smaller `max_depth` and leave the tree count unchanged. The forest depth moves from 12 to 8. The XGBoost depth moves from 6 to 4, and only if the forest was not far below. Score the new fit with the same 5 training folds and the same 28,248 validation claims, then print the validation MAE before and after that one change.

A close score or a higher score does not start a new fit in this cell.

In [15]:
# Compare the in-sample training MAE with the 5-fold MAE.
# Far below: training MAE is at least 10 percent lower than the fold MAE.
# Higher: training MAE is worse than the fold MAE.
# Close: training MAE sits between those two lines.

def training_fold_relation(train_mae, cv_mae):
    far_below_line = 0.90 * cv_mae
    if train_mae < far_below_line:
        return "far below"
    elif train_mae > cv_mae:
        return "higher"
    else:
        return "close"

# In-sample training MAE: the fitted model predicts the same 131,822 training claims.
# The 5-fold MAE stays the score from the earlier cell.
forest_train_mae = float(mean_absolute_error(y_train, forest_pipe.predict(X_train_raw)))
xgboost_train_mae = float(mean_absolute_error(y_train, xgboost_pipe.predict(X_train_raw)))

forest_relation = training_fold_relation(forest_train_mae, forest_cv_mae)
xgboost_relation = training_fold_relation(xgboost_train_mae, xgboost_cv_mae)

print(f"The random forest in-sample training MAE is {forest_train_mae} dollars. The 5-fold MAE is {forest_cv_mae} dollars. The training MAE is {forest_relation} the 5-fold MAE.")
print(f"The XGBoost in-sample training MAE is {xgboost_train_mae} dollars. The 5-fold MAE is {xgboost_cv_mae} dollars. The training MAE is {xgboost_relation} the 5-fold MAE.")

median_bar = 1827.3136338855847
print(f"The median bar is {median_bar} dollars. The forest 5-fold MAE is {forest_cv_mae} dollars. The XGBoost 5-fold MAE is {xgboost_cv_mae} dollars.")
print("Both 5-fold scores sit below the median bar, so this step does not add trees or raise the learning rate.")
print("Both first fits are scored, so this step does not repair a crash.")

# Lower complexity only when a training MAE is far below its 5-fold MAE.
# Change one knob on one model. Depth is the knob. Tree count stays the same.
if forest_relation == "far below": # change the random forest parameters if the training MAE is far below the 5-fold MAE
    model_to_change = "random forest"
    changed_knob = "max_depth"
    settings_2 = dict(forest_settings)
    settings_2["max_depth"] = 8
    model_2 = RandomForestRegressor(**settings_2, n_jobs=-1)
    before_val_mae = forest_val_mae
elif xgboost_relation == "far below": # change the XGBoost parameters if the training MAE is far below the 5-fold MAE
    model_to_change = "XGBoost"
    changed_knob = "max_depth"
    settings_2 = dict(xgboost_settings)
    settings_2["max_depth"] = 4
    model_2 = XGBRegressor(**settings_2, n_jobs=-1)
    before_val_mae = xgboost_val_mae
elif forest_relation == "close" or xgboost_relation == "close": # do not lower max_depth or the tree count if the training MAE is close to the 5-fold MAE
    model_to_change = "none"
    print("The training and 5-fold scores are close. This step does not lower max_depth or the tree count.")
else: # do not lower max_depth or the tree count if a training MAE is higher than its 5-fold MAE
    model_to_change = "none"
    print("A training MAE is higher than its 5-fold MAE. This step does not lower complexity.")

# Refit the model with the new parameters
if model_to_change != "none":
    pipeline_2 = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", model_2),
    ])
    fold_mae_2 = -cross_val_score(
        pipeline_2,
        X_train_raw,
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )
    cv_mae_2 = float(fold_mae_2.mean())
    pipeline_2.fit(X_train_raw, y_train)
    val_mae_2 = float(mean_absolute_error(y_val, pipeline_2.predict(X_val_raw)))

    print(f"The changed model is the {model_to_change}. The one knob changed is {changed_knob}.")
    print(f"The validation MAE before the change was {before_val_mae} dollars. The validation MAE after the change is {val_mae_2} dollars.")
    print(f"The 5-fold MAE after the change is {cv_mae_2} dollars.")

The random forest in-sample training MAE is 1223.9801792707158 dollars. The 5-fold MAE is 1282.8323805287268 dollars. The training MAE is close the 5-fold MAE.
The XGBoost in-sample training MAE is 1103.0595840165088 dollars. The 5-fold MAE is 1162.9594067528374 dollars. The training MAE is close the 5-fold MAE.
The median bar is 1827.3136338855847 dollars. The forest 5-fold MAE is 1282.8323805287268 dollars. The XGBoost 5-fold MAE is 1162.9594067528374 dollars.
Both 5-fold scores sit below the median bar, so this step does not add trees or raise the learning rate.
Both first fits are scored, so this step does not repair a crash.
The training and 5-fold scores are close. This step does not lower max_depth or the tree count.


In [16]:
# Keep the four Sprint 2 rows. Replace every earlier iteration-1 row with this one.
register = pd.read_csv(register_path)
register = register.loc[register["run_id"] != "iteration-1"].copy()

if model_to_change == "none":
    iteration_row = {
        "run_id": "iteration-1",
        "hypothesis": "A smaller max_depth is warranted only when the in-sample training MAE is far below the 5-fold MAE",
        "changed_variable": "none",
        "feature_set": feature_set,
        "preprocessing": preprocessing,
        "settings": "no refit",
        "seed": 1,
        "versions": versions,
        "runtime_seconds": 0,
        "train_or_cv_mae": "",
        "validation_mae": "",
        "validation_claims": 28248,
        "result": "not run",
        "interpretation": (
            f"Forest in-sample MAE is {forest_train_mae}. Forest 5-fold MAE is {forest_cv_mae}. Forest relation is {forest_relation}. "
            f"XGBoost in-sample MAE is {xgboost_train_mae}. XGBoost 5-fold MAE is {xgboost_cv_mae}. XGBoost relation is {xgboost_relation}. "
            f"Previous forest validation MAE is {forest_val_mae}. Previous XGBoost validation MAE is {xgboost_val_mae}. "
            "There is no new validation MAE."
        ),
        "next_decision": "Stop. The training-fold evidence did not call for one setting change. This sprint does not start another fit.",
    }
else:
    iteration_row = {
        "run_id": "iteration-1",
        "hypothesis": "A smaller max_depth can improve dollar MAE when training MAE is far below the 5-fold MAE",
        "changed_variable": f"{model_to_change} {changed_knob}",
        "feature_set": feature_set,
        "preprocessing": preprocessing,
        "settings": str(settings_2),
        "seed": 1,
        "versions": versions,
        "runtime_seconds": "",
        "train_or_cv_mae": cv_mae_2,
        "validation_mae": val_mae_2,
        "validation_claims": len(y_val),
        "result": "scored",
        "interpretation": (
            f"Previous validation MAE is {before_val_mae}. "
            f"New validation MAE is {val_mae_2}."
        ),
        "next_decision": (
            "Stop this sprint after this one change. A lower new MAE can be tested in a later sprint. "
            "The three-run cap is not a reason to keep tuning now."
            if val_mae_2 < before_val_mae
            else "Stop. The new validation MAE is not lower."
        ),
    }

register = pd.concat([register, pd.DataFrame([iteration_row])], ignore_index=True)
register.to_csv(register_path, index=False)


print(f"The expected value for iteration-1 rows is 1. The actual value for iteration-1 rows is {(register['run_id'] == 'iteration-1').sum()}.")
print(f"The expected value for register rows is 5. The actual value for register rows is {len(register)}.")
print(register.loc[register["run_id"] == "iteration-1", ["run_id", "changed_variable", "validation_mae", "result", "next_decision"]].to_string(index=False))


The expected value for iteration-1 rows is 1. The actual value for iteration-1 rows is 1.
The expected value for register rows is 5. The actual value for register rows is 5.
     run_id changed_variable validation_mae  result                                                                                                 next_decision
iteration-1             none                not run Stop. The training-fold evidence did not call for one setting change. This sprint does not start another fit.


Validation MAE comparison

- Write one row for each constant, each first model, and the iteration.
- The training or cross-validation MAE and the validation MAE are both in dollars.
- The validation score uses the same 28,248 claims. Test claims stay out of the table.
- A gap is this row's validation MAE minus a baseline validation MAE. A negative gap is a lower error.
- The iteration stays in the table. Its MAE and gap cells stay blank because that refit did not run.

In [17]:
# Creates a directory for the comparison table.
comparison_dir = (data_path.parents[1] / "artifacts/october/gate3-issue-reproducible-project/comparison")
comparison_dir.mkdir(parents=True, exist_ok=True)
comparison_path = comparison_dir / "validation_mae.csv"


register = pd.read_csv(register_path)
kept_runs = ["constant-mean", "constant-median", "forest-1", "iteraxgboost", "iteration-1"]
scored = register.loc[register["run_id"].isin(kept_runs)].copy()

# Sets the mean and median validation MAEs as baselines.
mean_validation_mae = float(scored.loc[scored["run_id"] == "constant-mean", "validation_mae"].iloc[0])
median_validation_mae = float(scored.loc[scored["run_id"] == "constant-median", "validation_mae"].iloc[0])

partition_labels = "train=131822, 5-fold inside training only; validation=28248; test held out; random_state=1"

# Creates a function to calculate the dollar gap.
def dollar_gap(validation_mae, baseline_mae):
    if pd.isna(validation_mae):
        return ""
    else:
        return float(validation_mae) - baseline_mae

# Creates a list of rows for the comparison table.
rows = []
for row in scored.itertuples(index=False):
    rows.append({
        "approach": row.run_id,
        "partition_labels": partition_labels,
        "train_or_cv_mae_dollars": "" if pd.isna(row.train_or_cv_mae) else float(row.train_or_cv_mae),
        "validation_mae_dollars": "" if pd.isna(row.validation_mae) else float(row.validation_mae),
        "validation_claims": int(row.validation_claims),
        "gap_vs_mean_baseline_dollars": dollar_gap(row.validation_mae, mean_validation_mae),
        "gap_vs_median_baseline_dollars": dollar_gap(row.validation_mae, median_validation_mae),
        "complexity": row.settings,
        "runtime_seconds": "" if pd.isna(row.runtime_seconds) else row.runtime_seconds,
    })

comparison = pd.DataFrame(rows)
comparison.to_csv(comparison_path, index=False)

print(f"The comparison file is {comparison_path}.")
print(comparison[["approach", "gap_vs_mean_baseline_dollars", "gap_vs_median_baseline_dollars"]].to_string(index=False))

print(f"The expected value for comparison rows is 5. The actual value for comparison rows is {len(comparison)}.")
print(f"The comparison file is {comparison_path}.")
print(comparison[["approach", "validation_mae_dollars", "gap_vs_mean_baseline_dollars", "gap_vs_median_baseline_dollars"]].to_string(index=False))


The comparison file is /Users/nevaehdickerson/Documents/MyGitHub/Allstate-1D-predicting-auto-claims-severity/artifacts/october/gate3-issue-reproducible-project/comparison/validation_mae.csv.
       approach gap_vs_mean_baseline_dollars gap_vs_median_baseline_dollars
  constant-mean                          0.0                     151.083413
constant-median                  -151.083413                            0.0
       forest-1                   -686.80584                    -535.722427
    iteration-1                                                            
The expected value for comparison rows is 5. The actual value for comparison rows is 4.
The comparison file is /Users/nevaehdickerson/Documents/MyGitHub/Allstate-1D-predicting-auto-claims-severity/artifacts/october/gate3-issue-reproducible-project/comparison/validation_mae.csv.
       approach validation_mae_dollars gap_vs_mean_baseline_dollars gap_vs_median_baseline_dollars
  constant-mean            1978.397047             

# High-loss tail

- The cut is the 99th percentile of training `loss`. Validation `loss` only places claims above or below that cut.
- Each approach reports the claim count and the MAE on the high-loss validation slice, and the claim count and the MAE on the remaining validation claims.
- The two claim counts add to 28,248. Those counts are the same for every approach because the cut uses the actual loss.
- `iteration-1` stays in the table. Its MAE cells stay blank because that refit did not run.

In [18]:
# The cut is the training 99th percentile of loss. Validation loss is not used to choose it.
high_loss_cut = float(np.quantile(y_train, 0.99))
high_loss_mask = y_val >= high_loss_cut
remaining_mask = y_val < high_loss_cut

high_loss_claims = int(high_loss_mask.sum())
remaining_claims = int(remaining_mask.sum())

print(f"The training 99th percentile of loss is {high_loss_cut} dollars.")
print(f"The expected value for validation claims is 28248. The actual value for high-loss claims plus remaining claims is {high_loss_claims + remaining_claims}.")
print(f"The high-loss validation claim count is {high_loss_claims}. The remaining validation claim count is {remaining_claims}.")

# MAE on one slice. The predictions follow the same validation row order as y_val.
def slice_mae(y_true, predictions, mask):
    if int(mask.sum()) == 0:
        return ""
    return float(mean_absolute_error(y_true[mask], predictions[mask]))

# The four scored approaches. The iteration has no new predictions.
approach_predictions = {
    "constant-mean": mean_predictions,
    "constant-median": median_predictions,
    "forest-1": forest_pipe.predict(X_val_raw),
    "xgboost-1": xgboost_pipe.predict(X_val_raw),
}

rows = []
for approach, predictions in approach_predictions.items():
    rows.append({
        "approach": approach,
        "training_99th_percentile_loss": high_loss_cut,
        "high_loss_claims": high_loss_claims,
        "high_loss_mae_dollars": slice_mae(y_val, predictions, high_loss_mask),
        "remaining_claims": remaining_claims,
        "remaining_mae_dollars": slice_mae(y_val, predictions, remaining_mask),
    })

rows.append({
    "approach": "iteration-1",
    "training_99th_percentile_loss": high_loss_cut,
    "high_loss_claims": high_loss_claims,
    "high_loss_mae_dollars": "",
    "remaining_claims": remaining_claims,
    "remaining_mae_dollars": "",
})


tail = pd.DataFrame(rows)
tail_path = comparison_dir / "high_loss_tail.csv"
tail.to_csv(tail_path, index=False)

# Prints the table.
print(f"The expected value for tail rows is 5. The actual value for tail rows is {len(tail)}.")
print(f"The tail file is {tail_path}.")
print(tail.to_string(index=False))



The training 99th percentile of loss is 14017.40979999999 dollars.
The expected value for validation claims is 28248. The actual value for high-loss claims plus remaining claims is 28248.
The high-loss validation claim count is 285. The remaining validation claim count is 27963.
The expected value for tail rows is 5. The actual value for tail rows is 5.
The tail file is /Users/nevaehdickerson/Documents/MyGitHub/Allstate-1D-predicting-auto-claims-severity/artifacts/october/gate3-issue-reproducible-project/comparison/high_loss_tail.csv.
       approach  training_99th_percentile_loss  high_loss_claims high_loss_mae_dollars  remaining_claims remaining_mae_dollars
  constant-mean                     14017.4098               285          15529.251114             27963           1840.286207
constant-median                     14017.4098               285          16449.184158             27963           1678.286952
       forest-1                     14017.4098               285           886

# High-loss tail note

The cut is the training 99th percentile of loss, $14,017.41. That slice has 285 validation claims. The other 27,963 validation claims are the remainder.

Both models have a lower overall validation MAE than both constants, and both models also have a lower MAE on the 285 large claims. On that comparison, overall error and large-claim error move together.

They do not move together between the two models. XGBoost has the lower overall validation MAE, $1,171.72 against the forest's $1,291.59. The forest has the lower large-claim MAE, $8,868.69 against XGBoost's $9,265.72. The same split appears between the constants: the median has the lower overall MAE, and the mean has the lower large-claim MAE.

This note does not select a model. The large-claim MAE is a check on 285 claims, not a reason to prefer the forest or XGBoost.

In [19]:
forest_val_pred = forest_pipe.predict(X_val_raw)
xgboost_val_pred = xgboost_pipe.predict(X_val_raw)

# Calculates the validation MAE gaps.
forest_fold_gap = forest_val_mae - forest_cv_mae
xgboost_fold_gap = xgboost_val_mae - xgboost_cv_mae
validation_gap = forest_val_mae - xgboost_val_mae

# Calculates the number of non-finite predictions.
mean_non_finite = int((~np.isfinite(mean_predictions)).sum())
median_non_finite = int((~np.isfinite(median_predictions)).sum())
forest_non_finite = int((~np.isfinite(forest_val_pred)).sum())
xgboost_non_finite = int((~np.isfinite(xgboost_val_pred)).sum())

# Prints the validation MAE gaps and the number of non-finite predictions.
print(f"The forest validation MAE minus the forest 5-fold MAE is {forest_fold_gap} dollars.")
print(f"The XGBoost validation MAE minus the XGBoost 5-fold MAE is {xgboost_fold_gap} dollars.")
print(f"The forest validation MAE minus the XGBoost validation MAE is {validation_gap} dollars.")
print(f"The expected value for non-finite mean predictions is 0. The actual value is {mean_non_finite}.")
print(f"The expected value for non-finite median predictions is 0. The actual value is {median_non_finite}.")
print(f"The expected value for non-finite forest predictions is 0. The actual value is {forest_non_finite}.")
print(f"The expected value for non-finite XGBoost predictions is 0. The actual value is {xgboost_non_finite}.")










The forest validation MAE minus the forest 5-fold MAE is 8.758826093426478 dollars.
The XGBoost validation MAE minus the XGBoost 5-fold MAE is 8.759575032792554 dollars.
The forest validation MAE minus the XGBoost validation MAE is 119.8722248365234 dollars.
The expected value for non-finite mean predictions is 0. The actual value is 0.
The expected value for non-finite median predictions is 0. The actual value is 0.
The expected value for non-finite forest predictions is 0. The actual value is 0.
The expected value for non-finite XGBoost predictions is 0. The actual value is 0.


# Name the winning model from the validation scores

- The four approaches and their validation MAEs, on the same 28,248 claims: training mean, training median, random forest, XGBoost.
- The naming rule: the lowest validation MAE, in original dollars.
- The named model, and the dollar gap against the forest and against the training median.
- The large-claim check beside the name: the training 99th-percentile cut, the 285 validation claims, and the forest and XGBoost misses on that slice.
- One sentence that the large-claim check is context and is not the naming rule.
- A statement that this uses the validation scores already computed above. No new fit. The test claims are not read.

In [20]:
validation_scores = pd.DataFrame([
    {"approach": "Training mean", "validation_mae": mean_mae},
    {"approach": "Training median", "validation_mae": median_mae},
    {"approach": "Random forest", "validation_mae": forest_val_mae},
    {"approach": "XGBoost", "validation_mae": xgboost_val_mae},
])

print(validation_scores)

          approach  validation_mae
0    Training mean     1978.397047
1  Training median     1827.313634
2    Random forest     1291.591207
3          XGBoost     1171.718982


# Gate 3 - Part 2
Document feature decisions and test a justified hypothesis if time permits, or record its deferral for November. Avoid unsupported feature removal or business and causal interpretations of anonymous predictors. Have a teammate independently verify the comparison and one model’s evaluation, recording limitations, disagreements, and artifact versions.



# Feature decision

- All 130 inputs stay in. No field is removed.
- A weak September correlation, or a correlation with another field, is not a removal reason.
- The deferred test is one XGBoost refit that drops one training-chosen field from the `cont11` and `cont12` pair. It is recorded in `artifacts/october/gate4-october-milestone-reproducible-project/handoff/november_handoff.md`.
- This deferral does not block Gate 3.